# End-to-End Model Customization with RHOAI

This notebook demonstrates the **complete lifecycle** of customizing a language model on Red Hat OpenShift AI (RHOAI) 3.4 — from raw documents to a fine-tuned model you can query.

### Pipeline Overview

```
Raw Documents
     │
     ▼
┌─────────────────────┐
│ 1. Document Prep    │  Docling: PDF/URL → structured Markdown
└─────────┬───────────┘
          │
          ▼
┌─────────────────────┐
│ 2. Data Generation  │  SDG Hub: documents → Q&A training pairs
│    (SDG Hub)        │  Uses teacher LLM to synthesize high-quality data
└─────────┬───────────┘
          │
          ▼
┌─────────────────────┐
│ 3. Data Formatting  │  Convert Q&A → Training Hub messages format
│    & Mixing         │  Add knowledge unmask flag, compute token stats
└─────────┬───────────┘
          │
          ▼
┌─────────────────────┐
│ 4. Model Training   │  Training Hub: SFT or OSFT fine-tuning
│    (Training Hub)   │  OSFT preserves base capabilities
└─────────┬───────────┘
          │
          ▼
┌─────────────────────┐
│ 5. Evaluation       │  Side-by-side: base model vs fine-tuned model
└─────────┬───────────┘
          │
          ▼
┌─────────────────────┐
│ 6. Serving          │  Deploy on RHOAI via KServe / vLLM
└─────────────────────┘
```

### Prerequisites

| Requirement | Details |
|-------------|--------|
| Python | 3.10+ |
| Teacher LLM | API endpoint (e.g. vLLM, OpenAI-compatible) for synthetic data generation |
| GPU | At least 1× A100/H100 for training; CPU-only for data generation |
| Packages | `sdg_hub[examples]`, `training_hub`, `transformers`, `docling` |

## 0. Environment Setup

In [ ]:
# Install required packages (uncomment if not already installed)
# !pip install sdg-hub[examples] training-hub transformers docling torch
# !pip install python-dotenv nest_asyncio pandas

In [ ]:
import os
import json
import time
from pathlib import Path

import pandas as pd
import nest_asyncio
from dotenv import load_dotenv

nest_asyncio.apply()
load_dotenv()

# ── Configuration ────────────────────────────────────────────────────────────
# Set these directly or in a .env file

TEACHER_MODEL = os.getenv("TEACHER_MODEL", "openai/gpt-oss-120b")
MODEL_API_KEY = os.getenv("MODEL_API_KEY", "")
MODEL_API_BASE = os.getenv("MODEL_API_BASE")  # e.g. https://your-vllm/v1

STUDENT_MODEL = os.getenv("STUDENT_MODEL", "meta-llama/Llama-3.1-8B-Instruct")

OUTPUT_DIR = Path(os.getenv("OUTPUT_DATA_FOLDER", "./output"))
CHECKPOINT_DIR = Path(os.getenv("CHECKPOINT_DIR", "./checkpoints"))

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

assert MODEL_API_KEY, "Set MODEL_API_KEY in .env or as an environment variable"

print(f"Teacher model : {TEACHER_MODEL}")
print(f"Student model : {STUDENT_MODEL}")
print(f"Output dir    : {OUTPUT_DIR}")

---
## 1. Document Preparation

Before we can generate training data, we need source documents in a structured format. [Docling](https://github.com/DS4SD/docling) converts PDFs, URLs, and other formats into clean Markdown with metadata.

If you already have pre-processed JSONL documents, skip to **Step 2**.

In [ ]:
from docling.document_converter import DocumentConverter

# Example: convert a PDF or URL to structured markdown
SOURCE_URLS = [
    # Replace with your own documents
    "https://arxiv.org/pdf/2305.14314",  # example: an arxiv paper
]

converter = DocumentConverter()
documents = []

for url in SOURCE_URLS:
    print(f"Processing: {url}")
    result = converter.convert(url)
    doc_md = result.document.export_to_markdown()
    documents.append({
        "document": doc_md,
        "domain": "technical",  # adjust per your domain
        "source": url,
        "document_outline": doc_md[:500],  # first 500 chars as outline
    })
    print(f"  → {len(doc_md)} characters extracted")

# Save pre-processed documents
doc_dir = OUTPUT_DIR / "documents"
doc_dir.mkdir(exist_ok=True)
doc_path = doc_dir / "source_documents.jsonl"

with open(doc_path, "w") as f:
    for doc in documents:
        f.write(json.dumps(doc) + "\n")

print(f"\nSaved {len(documents)} documents to {doc_path}")

---
## 2. Synthetic Data Generation (SDG Hub)

SDG Hub generates high-quality Q&A training pairs from your documents using a teacher LLM. It offers four knowledge flow variants, each producing different types of training data:

| Flow | What it generates |
|------|------------------|
| `extractive_summary` | Extracts key segments, generates QA from them |
| `detailed_summary` | Creates high-level summaries, derives QA pairs |
| `key_facts` | Decomposes into atomic facts with QA pairs |
| `doc_direct_qa` | Generates QA pairs directly from raw document |

Each flow is a YAML-defined pipeline of composable blocks.

In [ ]:
from sdg_hub import Flow
from datasets import load_dataset

# Load source documents as a HuggingFace dataset
doc_path = OUTPUT_DIR / "documents" / "source_documents.jsonl"
dataset = load_dataset("json", data_files=str(doc_path), split="train")
print(f"Loaded {len(dataset)} documents")
print(f"Columns: {dataset.column_names}")

In [ ]:
from sdg_hub import FlowRegistry

FlowRegistry.discover_flows()

FLOW_VARIANT_NAMES = {
    "extractive_summary": "Extractive Summary Knowledge Tuning Dataset Generation Flow",
    "detailed_summary":   "Detailed Summary Knowledge Tuning Dataset Generation Flow",
    "key_facts":          "Key Facts Knowledge Tuning Dataset Generation Flow",
    "doc_direct_qa":      "Document Based Knowledge Tuning Dataset Generation Flow",
}

# Pick which variants to run (all four, or a subset for faster iteration)
SELECTED_FLOWS = ["extractive_summary", "doc_direct_qa"]

generated_data = {}

for variant_name in SELECTED_FLOWS:
    flow_yaml = FlowRegistry.get_flow_path(FLOW_VARIANT_NAMES[variant_name])
    print(f"\n{'=' * 60}")
    print(f"Running flow: {variant_name}")
    print(f"  YAML: {flow_yaml}")
    print(f"{'=' * 60}")

    flow = Flow.from_yaml(flow_yaml)
    flow.set_model_config(
        model=TEACHER_MODEL,
        api_key=MODEL_API_KEY,
        api_base=MODEL_API_BASE,
    )

    start = time.time()
    result = flow.generate(
        dataset,
        checkpoint_dir=str(CHECKPOINT_DIR / variant_name),
    )
    elapsed = time.time() - start

    result_df = result.to_pandas() if hasattr(result, "to_pandas") else result

    out_path = OUTPUT_DIR / f"{variant_name}.jsonl"
    result_df.to_json(out_path, orient="records", lines=True)

    generated_data[variant_name] = result_df
    print(f"  → {len(result_df)} Q&A pairs generated in {elapsed:.1f}s")
    print(f"  → Saved to {out_path}")

In [ ]:
# Preview the generated data
for name, df in generated_data.items():
    print(f"\n── {name} ({len(df)} rows) ──")
    if "question" in df.columns and "response" in df.columns:
        sample = df.iloc[0]
        print(f"  Q: {sample['question'][:120]}...")
        print(f"  A: {sample['response'][:120]}...")
    else:
        print(f"  Columns: {list(df.columns)}")

---
## 3. Data Formatting & Mixing

Training Hub expects data in the **messages** chat format (OpenAI-style). For knowledge tuning, each sample is marked with `"unmask": true` so the loss is computed on all message roles except the system prompt. This is critical for knowledge absorption.

```json
{
  "messages": [
    {"role": "system", "content": "You are a knowledgeable assistant."},
    {"role": "user", "content": "What is ..."},
    {"role": "assistant", "content": "Based on ..."}
  ],
  "unmask": true
}
```

In [ ]:
SYSTEM_PROMPT = (
    "You are a knowledgeable assistant. Answer the user's question "
    "accurately and thoroughly based on your training data."
)

def qa_to_messages(question: str, answer: str) -> dict:
    """Convert a Q&A pair to Training Hub messages format."""
    return {
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": question},
            {"role": "assistant", "content": answer},
        ],
        "unmask": True,
    }


# Convert all generated Q&A into training format
training_samples = []

for name, df in generated_data.items():
    if "question" not in df.columns or "response" not in df.columns:
        print(f"Skipping {name}: missing question/response columns")
        continue
    for _, row in df.iterrows():
        sample = qa_to_messages(str(row["question"]), str(row["response"]))
        sample["source_flow"] = name
        training_samples.append(sample)

print(f"Total training samples: {len(training_samples)}")
print(f"  By source flow:")
for name in generated_data:
    count = sum(1 for s in training_samples if s.get("source_flow") == name)
    print(f"    {name}: {count}")

In [ ]:
# Compute token statistics (helps choose max_seq_len for training)
from transformers import AutoTokenizer

try:
    tokenizer = AutoTokenizer.from_pretrained(STUDENT_MODEL)
    token_counts = []

    for sample in training_samples:
        text = tokenizer.apply_chat_template(
            sample["messages"], tokenize=False, add_generation_prompt=False
        )
        n = len(tokenizer.encode(text))
        token_counts.append(n)

    s = pd.Series(token_counts)
    print(f"Token statistics ({STUDENT_MODEL} tokenizer):")
    print(f"  Mean   : {s.mean():.0f}")
    print(f"  Median : {s.median():.0f}")
    print(f"  P95    : {s.quantile(0.95):.0f}")
    print(f"  Max    : {s.max()}")
    print(f"  Total  : {s.sum():,}")
except Exception as e:
    print(f"Tokenization skipped: {e}")

In [ ]:
# Write the training JSONL file
mix_dir = OUTPUT_DIR / "training_mix"
mix_dir.mkdir(exist_ok=True)
train_path = mix_dir / "knowledge_train.jsonl"

with open(train_path, "w") as f:
    for sample in training_samples:
        record = {"messages": sample["messages"], "unmask": sample["unmask"]}
        f.write(json.dumps(record) + "\n")

print(f"Training data written to {train_path}")
print(f"  {len(training_samples)} samples")

# Preview a sample
print(f"\n── Sample training record ──")
print(json.dumps(training_samples[0], indent=2))

---
## 4. Model Training (Training Hub)

Training Hub provides three algorithms for knowledge tuning:

| Algorithm | Best for | Key property |
|-----------|----------|-------------|
| **SFT** | General fine-tuning | Full parameter update |
| **OSFT** | Knowledge injection | Preserves base model capabilities via orthogonal subspace constraint |
| **LoRA** | Memory-efficient fine-tuning | Trains low-rank adapter only; single L4 24GB GPU sufficient with QLoRA |

**OSFT is recommended** for knowledge tuning because it absorbs new domain knowledge while maintaining the model's existing instruction-following and reasoning abilities. **LoRA** is a good alternative when GPU memory is limited.

> **GPU Required:** SFT/OSFT require at least 1× A100 (80GB) for 8B models. LoRA with QLoRA (4-bit) works on 1× L4 24GB. Scale `nproc_per_node` for multi-GPU.

In [ ]:
# ── Training Configuration ────────────────────────────────────────────────

ALGORITHM = "osft"  # "sft", "osft", or "lora"

TRAINING_CONFIG = {
    "model_path": STUDENT_MODEL,
    "data_path": str(train_path),
    "ckpt_output_dir": str(CHECKPOINT_DIR / "knowledge_model"),
    "num_epochs": 4,
    "effective_batch_size": 32,
    "learning_rate": 2e-5,
    "max_seq_len": 4096,
    "nproc_per_node": 1,  # number of GPUs
}

print(f"Algorithm       : {ALGORITHM.upper()}")
for k, v in TRAINING_CONFIG.items():
    print(f"  {k:20s}: {v}")

In [ ]:
if ALGORITHM == "osft":
    from training_hub import osft

    osft(
        model_path=TRAINING_CONFIG["model_path"],
        data_path=TRAINING_CONFIG["data_path"],
        ckpt_output_dir=TRAINING_CONFIG["ckpt_output_dir"],
        unfreeze_rank_ratio=0.25,  # preserves general capability
        num_epochs=TRAINING_CONFIG["num_epochs"],
        effective_batch_size=TRAINING_CONFIG["effective_batch_size"],
        learning_rate=TRAINING_CONFIG["learning_rate"],
        max_seq_len=TRAINING_CONFIG["max_seq_len"],
        max_tokens_per_gpu=10000,
        unmask_messages=True,    # required for knowledge tuning
        use_liger=True,          # memory-efficient kernels
        nproc_per_node=TRAINING_CONFIG["nproc_per_node"],
    )

elif ALGORITHM == "sft":
    from training_hub import sft

    sft(
        model_path=TRAINING_CONFIG["model_path"],
        data_path=TRAINING_CONFIG["data_path"],
        ckpt_output_dir=TRAINING_CONFIG["ckpt_output_dir"],
        num_epochs=TRAINING_CONFIG["num_epochs"],
        effective_batch_size=TRAINING_CONFIG["effective_batch_size"],
        learning_rate=TRAINING_CONFIG["learning_rate"],
        max_seq_len=TRAINING_CONFIG["max_seq_len"],
        max_tokens_per_gpu=25000,
        nproc_per_node=TRAINING_CONFIG["nproc_per_node"],
    )

elif ALGORITHM == "lora":
    from training_hub import lora_sft

    lora_sft(
        model_path=TRAINING_CONFIG["model_path"],
        data_path=TRAINING_CONFIG["data_path"],
        ckpt_output_dir=TRAINING_CONFIG["ckpt_output_dir"],
        lora_r=16,
        lora_alpha=32,
        num_epochs=TRAINING_CONFIG["num_epochs"],
        effective_batch_size=TRAINING_CONFIG["effective_batch_size"],
        learning_rate=TRAINING_CONFIG["learning_rate"],
        load_in_4bit=True,
        max_seq_len=TRAINING_CONFIG["max_seq_len"],
        nproc_per_node=TRAINING_CONFIG["nproc_per_node"],
    )

else:
    raise ValueError(f"Unknown algorithm: {ALGORITHM}. Use 'sft', 'osft', or 'lora'.")

print(f"\nTraining complete. Checkpoint saved to {TRAINING_CONFIG['ckpt_output_dir']}")

---
## 5. Evaluation

Compare the **base model** and the **fine-tuned model** side by side on domain-specific test questions. The fine-tuned model should give more accurate, detailed answers on the knowledge you injected.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

TUNED_MODEL_PATH = TRAINING_CONFIG["ckpt_output_dir"]

def load_model(model_path, device="auto"):
    """Load a causal LM and its tokenizer."""
    print(f"Loading: {model_path}")
    tok = AutoTokenizer.from_pretrained(model_path)
    model = AutoModelForCausalLM.from_pretrained(
        model_path, torch_dtype=torch.bfloat16, device_map=device,
    )
    model.eval()
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    return model, tok


def generate_response(model, tokenizer, question, max_new_tokens=512):
    """Generate a response for a given question."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question},
    ]
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    generated = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()


# Load both models
base_model, base_tok = load_model(STUDENT_MODEL)
tuned_model, tuned_tok = load_model(TUNED_MODEL_PATH)

In [ ]:
# Test questions — replace with domain-specific questions for your use case
TEST_QUESTIONS = [
    "What are the key concepts discussed in the training documents?",
    "Explain the main methodology described in the source material.",
    "What are the practical applications of this knowledge?",
]

for i, question in enumerate(TEST_QUESTIONS, 1):
    print(f"\n{'=' * 80}")
    print(f"Question {i}: {question}")
    print("=" * 80)

    base_answer = generate_response(base_model, base_tok, question)
    tuned_answer = generate_response(tuned_model, tuned_tok, question)

    print(f"\n── Base Model ({STUDENT_MODEL}) ──")
    print(base_answer[:500])

    print(f"\n── Fine-Tuned Model ──")
    print(tuned_answer[:500])

print(f"\n{'=' * 80}")
print(f"Evaluation complete: {len(TEST_QUESTIONS)} questions compared.")

---
## 6. Serving on RHOAI

Once satisfied with the fine-tuned model, deploy it on RHOAI using KServe with the vLLM runtime.

### Option A: Upload to S3 and deploy via RHOAI Dashboard

```bash
# Upload the checkpoint to your S3-compatible storage
aws s3 cp --recursive ./checkpoints/knowledge_model s3://my-bucket/models/knowledge-model/
```

Then in the RHOAI dashboard:
1. **Model Registry** → Register the model with the S3 path
2. **Model Serving** → Create an InferenceService with the vLLM ServingRuntime
3. **Test** → Use the generated endpoint URL

### Option B: Deploy via KServe YAML

```yaml
apiVersion: serving.kserve.io/v1beta1
kind: InferenceService
metadata:
  name: knowledge-tuned-llama
  annotations:
    serving.kserve.io/deploymentMode: RawDeployment
spec:
  predictor:
    model:
      modelFormat:
        name: vLLM
      runtime: vllm-runtime
      storageUri: s3://my-bucket/models/knowledge-model/
      resources:
        limits:
          nvidia.com/gpu: 1
```

### Option C: Quick local test with vLLM

```bash
vllm serve ./checkpoints/knowledge_model \
    --port 8000 \
    --dtype bfloat16
```

Then query the served model:

In [ ]:
# Query the served model via OpenAI-compatible API
# (Uncomment after deploying with vLLM or KServe)

# import openai
#
# client = openai.OpenAI(
#     base_url="http://localhost:8000/v1",  # or your RHOAI inference endpoint
#     api_key="unused",
# )
#
# response = client.chat.completions.create(
#     model="knowledge-tuned-llama",
#     messages=[
#         {"role": "system", "content": "You are a knowledgeable assistant."},
#         {"role": "user", "content": "What are the key findings from the training data?"},
#     ],
#     max_tokens=512,
# )
#
# print(response.choices[0].message.content)

---
## Summary

This notebook walked through the complete RHOAI model customization lifecycle:

| Step | Tool | What happened |
|------|------|---------------|
| 1. Document Prep | Docling | Raw PDFs/URLs → structured Markdown |
| 2. Data Generation | SDG Hub | Documents → Q&A training pairs via teacher LLM |
| 3. Data Formatting | Custom | Q&A → Training Hub messages format with `unmask` |
| 4. Training | Training Hub | SFT or OSFT fine-tuning of student model |
| 5. Evaluation | Transformers | Side-by-side comparison base vs fine-tuned |
| 6. Serving | KServe + vLLM | Deploy on RHOAI for production inference |

### Key APIs used

```python
# SDG Hub — generate training data
flow = Flow.from_yaml("path/to/flow.yaml")
flow.set_model_config(model=..., api_key=..., api_base=...)
result = flow.generate(dataset)

# Training Hub — fine-tune the model
from training_hub import osft
osft(model_path=..., data_path=..., ckpt_output_dir=..., unmask_messages=True)
```

### Next steps

- **Scale up**: Add more source documents and run all four flow variants
- **Multi-GPU training**: Set `nproc_per_node` to your GPU count
- **KFP Pipeline**: Orchestrate this workflow as a Kubeflow Pipeline on RHOAI
- **Iterate**: Evaluate on domain-specific benchmarks and repeat the cycle